In [1]:
# clean nav history
import pandas as pd
from pathlib import Path

# Define paths
processed_path = Path("data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

# Load data
nav = pd.read_csv("../data/raw/02_nav_history.csv")
# Cleaning NAV history
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')

# Sort
nav = nav.sort_values(by=['amfi_code', 'date'])

# Forward fill NAV
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

# Remove duplicates
nav = nav.drop_duplicates()

# Remove invalid NAV
nav = nav[nav['nav'] > 0]

# Save
nav.to_csv(processed_path / "nav_history_clean.csv", index=False)


In [2]:
# CLEAN INVESTOR TRANSACTIONS

import pandas as pd
from pathlib import Path
# Paths
raw_path = Path("data/raw")
processed_path = Path("data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

# Load Dataset

txn = pd.read_csv("../data/raw/08_investor_transactions.csv")


print("ORIGINAL SHAPE:")
print(txn.shape)

print("\nFIRST 5 ROWS:")
print(txn.head())

print("\nCOLUMN NAMES:")
print(txn.columns.tolist())


# Transaction Type Cleaning
#
if 'transaction_type' in txn.columns:

    txn['transaction_type'] = (
        txn['transaction_type']
        .astype(str)
        .str.strip()
        .str.upper()
    )

    txn['transaction_type'] = txn[
        'transaction_type'
    ].replace({
        'SYSTEMATIC INVESTMENT': 'SIP',
        'SIP': 'SIP',
        'LUMPSUM': 'LUMPSUM',
        'REDEMPTION': 'REDEMPTION'
    })

# Amount Cleaning
if 'amount' in txn.columns:

    txn['amount'] = pd.to_numeric(
        txn['amount'],
        errors='coerce'
    )

    txn = txn[
        txn['amount'] > 0
    ]

# Date Cleaning

if 'transaction_date' in txn.columns:

    txn['transaction_date'] = pd.to_datetime(
        txn['transaction_date'],
        errors='coerce'
    )

# KYC Cleaning
if 'kyc_status' in txn.columns:

    txn['kyc_status'] = (
        txn['kyc_status']
        .astype(str)
        .str.strip()
        .str.upper()
    )

    txn = txn[
        txn['kyc_status'].isin(
            ['VERIFIED', 'PENDING', 'REJECTED']
        )
    ]

# Remove Duplicates
txn = txn.drop_duplicates()

# Remove Fully Empty Rows
txn = txn.dropna(how='all')

# Save Clean File

txn.to_csv(
    processed_path /
    "investor_transactions_clean.csv",
    index=False
)

# Verification
print("\nCLEANED SHAPE:")
print(txn.shape)

print("\nNULL VALUES:")
print(txn.isnull().sum())

print(
    "\nSUCCESS: "
    "investor_transactions_clean.csv SAVED"
)

ORIGINAL SHAPE:
(32778, 13)

FIRST 5 ROWS:
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Veri

In [3]:
# CLEAN INVESTOR TRANSACTIONS

# Convert transaction date
txn['transaction_date'] = pd.to_datetime(
    txn['transaction_date'],
    errors='coerce'
)

# Standardize transaction type
txn['transaction_type'] = (
    txn['transaction_type']
    .astype(str)
    .str.strip()
    .str.upper()
)

txn['transaction_type'] = (
    txn['transaction_type']
    .replace({
        'SYSTEMATIC INVESTMENT': 'SIP',
        'SYSTEMATIC INVESTMENT PLAN': 'SIP',
        'SIP': 'SIP',
        'LUMPSUM': 'LUMPSUM',
        'LUMP SUM': 'LUMPSUM',
        'REDEMPTION': 'REDEMPTION'
    })
)

# Convert amount to numeric
txn['amount'] = pd.to_numeric(
    txn['amount_inr'],
    errors='coerce'
)

# Remove invalid amounts
txn = txn[txn['amount'] > 0]

# Standardize KYC status
txn['kyc_status'] = (
    txn['kyc_status']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Remove duplicates
txn = txn.drop_duplicates()

# Save
txn.to_csv(
    processed_path / "investor_transactions_clean.csv",
    index=False
)

# Verification
print(txn.head())

print("\nDATA TYPES:")
print(txn.dtypes)

print("\nCLEANED SHAPE:")
print(txn.shape)

print(
    "\nSUCCESS: investor_transactions_clean.csv SAVED"
)

  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       REDEMPTION      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          LUMPSUM        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  amount  
0          UPI   VERIFIED    1834  
1       Cheque   VER

In [4]:
# DAY 2 — CLEAN SCHEME PERFORMANCE

import pandas as pd
from pathlib import Path

# Paths
raw_path = Path("data/raw")
processed_path = Path("data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

# Load dataset

perf = pd.read_csv("../data/raw/07_scheme_performance.csv")

print("ORIGINAL SHAPE:")
print(perf.shape)

print("\nFIRST 5 ROWS:")
print(perf.head())

# Convert numeric columns

numeric_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'benchmark_3yr_pct',
    'alpha',
    'beta',
    'sharpe_ratio',
    'sortino_ratio'
]

for col in numeric_cols:

    if col in perf.columns:

        perf[col] = pd.to_numeric(
            perf[col],
            errors='coerce'
        )

# Remove duplicates

perf = perf.drop_duplicates()

# Remove invalid return values

if 'return_1yr_pct' in perf.columns:
    perf = perf[
        perf['return_1yr_pct'].notna()
    ]

# Save cleaned file

perf.to_csv(
    processed_path /
    "scheme_performance_clean.csv",
    index=False
)

# Verification

print("\nNULL VALUES:")
print(perf.isnull().sum())

print("\nCLEANED SHAPE:")
print(perf.shape)

print("\nDATA TYPES:")
print(perf.dtypes)

print(
    "\nSUCCESS: scheme_performance_clean.csv SAVED"
)

ORIGINAL SHAPE:
(40, 19)

FIRST 5 ROWS:
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_rati